<a href="https://colab.research.google.com/github/adane21-png/Challenges-in-Representation-Learning/blob/main/fer2013_experiments.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🧠 FER2013 – Facial Expression Recognition

**Kaggle Competition:** [Challenges in Representation Learning: Facial Expression Recognition Challenge](https://www.kaggle.com/competitions/challenges-in-representation-learning-facial-expression-recognition-challenge)

**WandB Project:** [fer2013-experiments](https://wandb.ai)

---

### Experiment Overview

| Exp | Model | Expected Behavior |
|-----|-------|-------------------|
| 01 | TinyFERNet (no aug) | **Underfitting** |
| 02 | TinyFERNet + augmentation | Underfitting (less) |
| 03 | MediumFERNet (no regularization) | **Overfitting** |
| 04 | MediumFERNet + Dropout + aug | Balanced |
| 05 | MediumFERNet + SGD | Optimizer comparison |
| 06 | DeepFERNet | More capacity |
| 07 | DeepFERNet + class weights | Handle class imbalance |
| 08 | ResNet18 (frozen backbone) | Transfer learning limit |
| 09 | ResNet18 (full fine-tune) | **Best result** |

In [ ]:
# ── 1. Install dependencies ──────────────────────────────────────────────────
!pip install -q wandb kaggle

In [ ]:
# ── 2. WandB login ───────────────────────────────────────────────────────────
import wandb
wandb.login()  # Enter your API key from wandb.ai/authorize when prompted

In [ ]:
# ── 3. Kaggle credentials & dataset download ─────────────────────────────────
import os
from google.colab import userdata

# Save your kaggle.json contents as KAGGLE_JSON in Colab Secrets (key icon on left)
os.makedirs('/root/.kaggle', exist_ok=True)
with open('/root/.kaggle/kaggle.json', 'w') as f:
    f.write(userdata.get('KAGGLE_JSON'))
os.chmod('/root/.kaggle/kaggle.json', 0o600)

!kaggle competitions download -c challenges-in-representation-learning-facial-expression-recognition-challenge
!unzip -q '*.zip' 2>/dev/null || true
!ls *.csv

In [ ]:
# ── 4. Clone repo & copy data ────────────────────────────────────────────────
GITHUB_USER = 'adane21-png'
REPO_NAME   = 'Challenges-in-Representation-Learning'

!git clone https://github.com/{GITHUB_USER}/{REPO_NAME}.git fer2013_repo
%cd fer2013_repo

!cp /content/train.csv .
!cp /content/test.csv  .

In [ ]:
# ── 5. Data exploration ──────────────────────────────────────────────────────
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

df = pd.read_csv('train.csv')
print(f"Dataset shape: {df.shape}")
print(f"\nSplit distribution:")
print(df['Usage'].value_counts())
print(f"\nClass distribution:")
print(df['emotion'].value_counts().sort_index())

EMOTIONS = {0:'Angry', 1:'Disgust', 2:'Fear', 3:'Happy', 4:'Sad', 5:'Surprise', 6:'Neutral'}

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Bar chart
counts = df[df['Usage']=='Training']['emotion'].value_counts().sort_index()
colors = ['#e74c3c','#8e44ad','#3498db','#f1c40f','#2ecc71','#e67e22','#95a5a6']
axes[0].bar([EMOTIONS[i] for i in counts.index], counts.values, color=colors)
axes[0].set_title('Class Distribution (Training)', fontsize=12)
axes[0].tick_params(axis='x', rotation=30)
axes[0].set_ylabel('Number of images')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 50, str(v), ha='center', fontsize=9)

# Pie chart
axes[1].pie(counts.values, labels=[EMOTIONS[i] for i in counts.index],
            autopct='%1.1f%%', colors=colors)
axes[1].set_title('Class Proportions', fontsize=12)

plt.suptitle('FER2013 Dataset Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Sample images per class
fig2, axes2 = plt.subplots(2, 7, figsize=(14, 5))
for cls in range(7):
    for row, sample_idx in enumerate([0, 10]):
        sample = df[(df['emotion']==cls) & (df['Usage']=='Training')].iloc[sample_idx]
        pixels = np.array(sample['pixels'].split(), dtype=np.float32).reshape(48, 48)
        axes2[row][cls].imshow(pixels, cmap='gray')
        if row == 0:
            axes2[row][cls].set_title(EMOTIONS[cls], fontsize=9, fontweight='bold')
        axes2[row][cls].axis('off')
plt.suptitle('Sample Images per Class (2 per class)', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# ── 6. Run ALL experiments (auto-logged to WandB) ────────────────────────────
WANDB_PROJECT = 'fer2013-experiments'

!python src/train.py \
    --data_path train.csv \
    --wandb_project {WANDB_PROJECT}

In [ ]:
# ── 7. Run a SINGLE experiment by index (0–8) ────────────────────────────────
# Useful for re-running or debugging a specific experiment
# 0=tiny_baseline, 1=tiny_aug, 2=medium_no_reg, 3=medium_reg,
# 4=medium_sgd, 5=deep, 6=deep_weights, 7=resnet18_frozen, 8=resnet18_finetune

EXP_IDX = 5

!python src/train.py \
    --data_path train.csv \
    --wandb_project fer2013-experiments \
    --exp_idx {EXP_IDX}

In [ ]:
# ── 8. Generate Kaggle submission from best model ────────────────────────────
import torch, sys
import pandas as pd
import numpy as np
sys.path.insert(0, 'src')
from train import ARCHITECTURES, get_transforms, FER2013Dataset
from torch.utils.data import DataLoader

# Update these to match your best experiment
BEST_ARCH       = 'resnet18'
BEST_CHECKPOINT = 'best_exp09_resnet18_finetune.pt'
ARCH_KWARGS     = {'freeze_backbone': False}
IMG_SIZE        = 64

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

model = ARCHITECTURES[BEST_ARCH](num_classes=7, **ARCH_KWARGS).to(device)
model.load_state_dict(torch.load(BEST_CHECKPOINT, map_location=device))
model.eval()

test_df = pd.read_csv('test.csv')
if 'emotion' not in test_df.columns:
    test_df['emotion'] = 0
if 'Usage' not in test_df.columns:
    test_df['Usage'] = 'Training'

_, val_tf = get_transforms(augment=False, img_size=IMG_SIZE)
test_ds     = FER2013Dataset(test_df, transform=val_tf, split='Training')
test_loader = DataLoader(test_ds, batch_size=64, shuffle=False)

all_preds = []
with torch.no_grad():
    for imgs, _ in test_loader:
        preds = model(imgs.to(device)).argmax(1).cpu().numpy()
        all_preds.extend(preds)

submission = pd.DataFrame({'emotion': all_preds})
submission.to_csv('submission.csv', index=False)
print(f"Submission saved: {len(submission)} predictions")
print(submission['emotion'].value_counts().sort_index())

In [ ]:
# ── 9. Push results back to GitHub ───────────────────────────────────────────
from google.colab import userdata
GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
GITHUB_USER  = 'adane21-png'
REPO_NAME    = 'Challenges-in-Representation-Learning' #

!git config user.email 'colab@fer2013.com'
!git config user.name  'FER2013 Experiments'
!git add submission.csv
!git commit -m 'feat: add kaggle submission from best model' --allow-empty
!git push https://{GITHUB_TOKEN}@github.com/{GITHUB_USER}/{REPO_NAME}.git main
print("Successfully pushed to GitHub!")